# Training Notebook 2025H1 (Spark ML, Distributed)

This notebook trains Spark ML baselines on pre-engineered features and saves metrics/predictions to HDFS.

In [13]:
import json
import urllib.request
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("DemandPredictionTrainingSparkML2025H1")
    .master("spark://master:7077")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def cluster_util(stage_name):
    print(f"\n===== CLUSTER UTILIZATION: {stage_name} =====")
    try:
        payload = json.load(urllib.request.urlopen("http://master:8080/json/"))
        workers = payload.get("workers", [])
        print("alive workers:", payload.get("aliveworkers"))
        print("active apps :", len(payload.get("activeapps", [])))
        for w in workers:
            print("worker", w.get("id"), "cores", f"{w.get('coresused', 0)}/{w.get('cores', 0)}", "memory", f"{w.get('memoryused', 0)}/{w.get('memory', 0)}")
    except Exception as e:
        print("Could not query Spark Master JSON:", e)

cluster_util("session_started")


===== CLUSTER UTILIZATION: session_started =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048


In [14]:
FEATURE_PATH = "/user/data/feature_engineering/demand_prediction_2025H1_features"
OUT_BASE = "/user/data/results/demand_prediction/2025H1_sparkml"
TARGET_COL = "pickup_demand"

# Enforce the intended training window (Sep-Nov 2025).
TRAIN_START_TS = "2025-09-01 00:00:00"
TRAIN_END_TS = "2025-12-01 00:00:00"

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "lag_144", "lag_1008",
    "roll_mean_6", "roll_mean_18", "roll_std_18",
]

df_all = spark.read.parquet(FEATURE_PATH)
if "pickup_bin_10m" not in df_all.columns:
    raise ValueError("Missing required column: pickup_bin_10m")

df = (
    df_all
    .where((F.col("pickup_bin_10m") >= F.to_timestamp(F.lit(TRAIN_START_TS))) & (F.col("pickup_bin_10m") < F.to_timestamp(F.lit(TRAIN_END_TS))))
    .cache()
)

rows = df.count()
if rows == 0:
    raise ValueError(
        f"No rows found in requested training window [{TRAIN_START_TS}, {TRAIN_END_TS})."
    )

print(f"Feature rows in training window [{TRAIN_START_TS}, {TRAIN_END_TS}):", rows)
df.groupBy("split").count().show()
cluster_util("after_feature_read")

# Rebuild split inside the filtered window to avoid empty train/test partitions.
bounds = df.agg(F.min("pickup_bin_10m").alias("min_ts"), F.max("pickup_bin_10m").alias("max_ts")).first()
if bounds["min_ts"] is None or bounds["max_ts"] is None:
    raise ValueError("Cannot compute local split bounds.")

min_epoch = int(bounds["min_ts"].timestamp())
max_epoch = int(bounds["max_ts"].timestamp())
cut_epoch = int(min_epoch + 0.7 * (max_epoch - min_epoch))

df = df.withColumn(
    "split_local",
    F.when(
        F.col("pickup_bin_10m") < F.to_timestamp(F.from_unixtime(F.lit(cut_epoch))),
        F.lit("train")
    ).otherwise(F.lit("test"))
).cache()

train_df = df.where(F.col("split_local") == "train")
test_df = df.where(F.col("split_local") == "test")
print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

Feature rows in training window [2025-09-01 00:00:00, 2025-12-01 00:00:00): 3433248


+-----+-------+
|split|  count|
+-----+-------+
| test|3433248|
+-----+-------+


===== CLUSTER UTILIZATION: after_feature_read =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048


Train rows: 2403326
Test rows : 1029922


In [15]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

# Fast mode is recommended for this cluster size (2 workers x 2 cores).
FAST_MODE = True
TRAIN_SAMPLE_FRACTION = 0.35 if FAST_MODE else 1.0

train_fit_df = train_df
if TRAIN_SAMPLE_FRACTION < 1.0:
    train_fit_df = train_df.sample(withReplacement=False, fraction=TRAIN_SAMPLE_FRACTION, seed=42).cache()
    print(f"[FAST_MODE] Using sampled train rows: {train_fit_df.count()} / {train_df.count()}")

models = [
    (
        "linear_regression",
        LinearRegression(
            featuresCol="features",
            labelCol=TARGET_COL,
            predictionCol="prediction",
            maxIter=60 if FAST_MODE else 100,
            regParam=0.1,
            elasticNetParam=0.0,
        ),
    ),
    (
        "random_forest",
        RandomForestRegressor(
            featuresCol="features",
            labelCol=TARGET_COL,
            predictionCol="prediction",
            numTrees=80 if FAST_MODE else 180,
            maxDepth=10 if FAST_MODE else 12,
            seed=42,
        ),
    ),
    (
        "gbt",
        GBTRegressor(
            featuresCol="features",
            labelCol=TARGET_COL,
            predictionCol="prediction",
            maxIter=50 if FAST_MODE else 120,
            maxDepth=5 if FAST_MODE else 6,
            stepSize=0.05,
            seed=42,
        ),
    ),
]

mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")

metrics_rows = []
pred_union = None

for model_name, reg in models:
    cluster_util(f"before_train_{model_name}")
    pipeline = Pipeline(stages=[assembler, reg])
    fitted = pipeline.fit(train_fit_df)
    pred = fitted.transform(test_df).withColumn(
        "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
    )

    mae_val = float(mae_eval.evaluate(pred))
    rmse_val = float(rmse_eval.evaluate(pred))
    mape_val = (
        pred.where(F.col(TARGET_COL) > 0)
        .select((F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)).alias("ape"))
        .agg((F.avg(F.col("ape")) * 100.0).alias("mape"))
        .first()["mape"]
    )

    metrics_rows.append({
        "model": model_name,
        "MAE": mae_val,
        "RMSE": rmse_val,
        "MAPE": float(mape_val) if mape_val is not None else None,
    })

    slim_pred = pred.select(
        F.lit(model_name).alias("model"),
        "PULocationID",
        "pickup_bin_10m",
        TARGET_COL,
        "prediction",
    )
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)
    cluster_util(f"after_train_{model_name}")

metrics_pdf = pd.DataFrame(metrics_rows).sort_values("MAPE")
display(metrics_pdf)
best_model_name = metrics_pdf.iloc[0]["model"]
print("Best model by MAPE:", best_model_name)

[FAST_MODE] Using sampled train rows: 842542 / 2403326

===== CLUSTER UTILIZATION: before_train_linear_regression =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048



===== CLUSTER UTILIZATION: after_train_linear_regression =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048

===== CLUSTER UTILIZATION: before_train_random_forest =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048


26/04/03 03:16:20 WARN DAGScheduler: Broadcasting large task binary with size 1682.5 KiB
26/04/03 03:16:25 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/04/03 03:16:31 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/04/03 03:16:37 WARN DAGScheduler: Broadcasting large task binary with size 1890.4 KiB
26/04/03 03:16:38 WARN DAGScheduler: Broadcasting large task binary with size 11.9 MiB
26/04/03 03:16:48 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB



===== CLUSTER UTILIZATION: after_train_random_forest =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048

===== CLUSTER UTILIZATION: before_train_gbt =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048



===== CLUSTER UTILIZATION: after_train_gbt =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048


,model,MAE,RMSE,MAPE
0,linear_regression,1.111713,2.668239,49.254960
1,random_forest,1.192982,3.522031,49.977038
2,gbt,1.209722,3.544097,51.072407


Best model by MAPE: linear_regression


In [16]:
metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")

print("Saved outputs:")
print(f"- {OUT_BASE}/metrics")
print(f"- {OUT_BASE}/predictions")
metrics_sdf.orderBy(F.col("MAPE").asc_nulls_last()).show(truncate=False)
cluster_util("after_output_write")

Saved outputs:
- /user/data/results/demand_prediction/2025H1_sparkml/metrics
- /user/data/results/demand_prediction/2025H1_sparkml/predictions
+------------------+-----------------+-----------------+-----------------+
|MAE               |MAPE             |RMSE             |model            |
+------------------+-----------------+-----------------+-----------------+
|1.1117127919526006|49.25495993238293|2.668239461697362|linear_regression|
|1.1929815421228893|49.9770381950957 |3.522030699013151|random_forest    |
|1.209721915178623 |51.07240693333567|3.544097277765641|gbt              |
+------------------+-----------------+-----------------+-----------------+


===== CLUSTER UTILIZATION: after_output_write =====
alive workers: 2
active apps : 1
worker worker-20260403000927-172.18.0.3-7078 cores 2/2 memory 2048/2048
worker worker-20260403000927-172.18.0.4-7078 cores 2/2 memory 2048/2048


In [17]:
# Optional cleanup: run this when training is done.
spark.catalog.clearCache()
spark.stop()
print("Training Spark session stopped.")

Training Spark session stopped.
